# 9c. Routage et Repli : choisir le bon modèle, au bon moment

Ce notebook poursuit [09_Production_Patterns.ipynb](09_Production_Patterns.ipynb), qui traitait des
patterns d'appel à une API donnée (retry, streaming, rate limiting). Ici, la question change
d'échelle : **quand on dispose de plusieurs modèles et de plusieurs fournisseurs, qui répond à
quelle question — et que fait-on quand l'un d'eux tombe ?**

Trois briques de production :

1. **Le routage par politique** — router chaque question vers un modèle selon son coût, la
   difficulté estimée de la question et un budget par requête ;
2. **La cascade petit-vers-grand** — tenter d'abord le modèle économique, n'escalader vers le
   modèle coûteux que quand la réponse semble douteuse ;
3. **Le repli fournisseur** — survivre à la panne d'un fournisseur en basant sur le suivant.

Ce sont trois réponses au même problème vu sous trois angles : décider **avant** de poser la
question (politique), décider **après** une première réponse (cascade), décider **pendant** un
incident (repli). Le fil du notebook les construit dans cet ordre, chacune mesurée sur le même
étalon — une banque de cent questions à réponse exacte — pour que les trois verdicts se
comparent. Et la mesure a une discipline constante : chaque stratégie est jugée sur les trois
monnaies (exactitude, latence, coût), jamais sur une seule.

> **Terrain réel.** La flotte fait tourner un proxy multi-fournisseurs (OpenRouter, qui expose son
> catalogue et ses prix via `/v1/models`) et un service d'inférence local (Ollama sur ce poste).
> Ce notebook **lit** ce que ce terrain expose — catalogue, prix, latences — et ne touche jamais à
> sa configuration, qui appartient à un autre workspace.

> **Clés API.** Les clés vivent dans `.env` (cf. `00-GenAI-Environment`), chargées par
> `python-dotenv`. Aucune clé n'est jamais écrite ni affichée dans ce notebook.

## Vérification de l'environnement


In [1]:
import os, json, time, random, re, statistics
from pathlib import Path
from dotenv import load_dotenv

# Le .env de la série GenAI (deux niveaux au-dessus du dossier Texte/)
load_dotenv(Path("..") / ".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("OPENROUTER_API_KEY :", "présente" if OPENROUTER_API_KEY else "ABSENTE -- sections proxy limitées")
print("OPENAI_API_KEY    :", "présente" if OPENAI_API_KEY else "absente")

# Reproductibilité : la banque de questions est générée avec une graine fixe
random.seed(20260928)


OPENROUTER_API_KEY : présente
OPENAI_API_KEY    : présente


In [2]:
# Sondage du terrain, en LECTURE SEULE : on interroge les catalogues, on ne configure rien
import urllib.request

def probe_catalogue(nom, url):
    req = urllib.request.Request(url)
    t0 = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            data = json.loads(r.read().decode("utf-8"))
        dt = (time.perf_counter() - t0) * 1000
        modeles = [m["id"] for m in data.get("data", [])]
        print(f"{nom:38s} HTTP 200  {len(modeles):4d} modèles  ({dt:.0f} ms)")
        return modeles
    except Exception as e:
        print(f"{nom:38s} INJOIGNABLE ({type(e).__name__})")
        return []

ollama_modeles = probe_catalogue("Inference locale (Ollama)", "http://127.0.0.1:11434/v1/models")
or_modeles = probe_catalogue("Proxy multi-fournisseurs (OpenRouter)",
                             "https://openrouter.ai/api/v1/models")


Inference locale (Ollama)              HTTP 200     2 modèles  (93 ms)


Proxy multi-fournisseurs (OpenRouter)  HTTP 200   458 modèles  (184 ms)


### Lecture du terrain

Deux fournisseurs vivants répondent, et l'asymétrie est la donnée centrale : **2 modèles** côté
inférence locale contre **458** côté proxy multi-fournisseurs. Le sondage a coûté **93 ms** au
premier et **184 ms** au second — deux réponses HTTP 200 obtenues en moins de cinquième de
seconde : le terrain est **mesurable à chaud**, pas documenté à froid. Cette asymétrie structure
tout le notebook : le local est étroit mais gratuit, immédiat et maîtrisé ; le proxy est un
catalogue entier, tarifé au token, dont la diversité est la matière première de toute stratégie
de choix. On notera ce que le sondage **ne fait pas** : il interroge les catalogues en lecture
seule, sans jamais reconfigurer le terrain — chaque chiffre de la suite est une observation
reproductible, pas une commande envoyée. C'est la discipline qui permettra de comparer deux
exécutions du notebook sans craindre d'avoir changé le terrain entre-temps.
Un mot sur la reproductibilité de ce sondage : les catalogues sont relus à chaque exécution,
et le notebook n'enregistre que ce qu'ils **ont répondu** — si le proxy expose demain 470
modèles, la ré-exécution le dira sans que rien d'autre ne bouge. La comparaison entre exécutions
reste ainsi honnête : le terrain peut vivre, le protocole ne change pas.

## 1. Le catalogue et ses prix : la matière première du routeur

Un routeur qui arbitre sur le coût a besoin des coûts — sinon il arbitre sur des impressions.
L'API `/v1/models` du proxy renvoie, pour chaque modèle, son prix par token d'entrée et de
sortie : le terrain publie lui-même la table de décision, et c'est une propriété remarquable du
proxy multi-fournisseurs — la donnée économique voyage avec la donnée technique. On y ajoute le
modèle local, dont le prix marginal est nul mais dont les ressources sont bornées au matériel
présent.

Trois choix de lecture structureront la suite : les prix sont **relus à chaque exécution** (un
catalogue figé est un catalogue périmé — OpenRouter fait varier ses prix) ; on ne retient que
quelques candidats (quatre lignes se lisent, 458 se compilent) ; et le **contexte** fait partie
du prix — un modèle gratuit à contexte étroit n'achète pas la même chose qu'un modèle gratuit à
contexte large. La question que cette section prépare : quand le grand modèle est gratuit,
qu'est-ce qui reste à arbitrer ?

In [3]:
req = urllib.request.Request("https://openrouter.ai/api/v1/models")
with urllib.request.urlopen(req, timeout=15) as r:
    catalogue = {m["id"]: m for m in json.loads(r.read().decode("utf-8"))["data"]}

def prix_par_million(m):
    """Prix USD par million de tokens (entrée, sortie) — 0.0 pour les modèles :free."""
    p = m.get("pricing", {})
    try:
        return float(p.get("prompt", "1")) * 1e6, float(p.get("completion", "1")) * 1e6
    except (TypeError, ValueError):
        return float("nan"), float("nan")

# Candidats retenus pour la démonstration — choisis dans le catalogue RÉEL sondé ci-dessus
CANDIDATS_OR = [
    "mistralai/mistral-nemo",           # payant, quasi gratuit
    "meta-llama/llama-3.1-8b-instruct", # payant, petit
    "google/gemma-4-31b-it:free",       # grand, gratuit
]

catalogue_prix = {}
print(f"{'modèle':36s} {'$in/M':>9s} {'$out/M':>9s} {'ctx':>8s}  fournisseur")
for mid in CANDIDATS_OR:
    m = catalogue[mid]
    pin, pout = prix_par_million(m)
    catalogue_prix[mid] = {"in": pin, "out": pout, "ctx": m.get("context_length", 0)}
    print(f"{mid:36s} {pin:9.3f} {pout:9.3f} {m.get('context_length', 0):8d}  openrouter")

# Le modèle local : gratuit, mais limité au matériel présent
LOCAL_ID = "qwen2.5:7b"
LOCAL_KEY = f"local/{LOCAL_ID}"
catalogue_prix[LOCAL_KEY] = {"in": 0.0, "out": 0.0, "ctx": 32768}
print(f"{LOCAL_KEY:36s} {0.0:9.3f} {0.0:9.3f} {32768:8d}  ollama (local)")


modèle                                   $in/M    $out/M      ctx  fournisseur


mistralai/mistral-nemo                   0.019     0.030   131072  openrouter
meta-llama/llama-3.1-8b-instruct         0.050     0.080   131072  openrouter
google/gemma-4-31b-it:free               0.000     0.000   262144  openrouter
local/qwen2.5:7b                         0.000     0.000    32768  ollama (local)


### Lecture du catalogue

Quatre lignes, trois enseignements. D'abord l'écart de prix **réel mais modeste** entre les
payants : `mistral-nemo` à **$0.019/$0.030** par million (entrée/sortie) contre
`llama-3.1-8b-instruct` à **$0.050/$0.080** — noter le rapport sortie/entrée d'environ 1,6 pour
chacun : produire un token coûte plus cher que le lire, et c'est la longueur des réponses qui
domine la facture, pas celle des questions. Ensuite la ligne décisive : **un grand modèle
gratuit existe** (`gemma-4-31b-it:free`, $0.000/$0.000) avec un contexte de **262144** tokens —
huit fois les 32768 du local. Le routage de ce notebook n'est donc pas « gratuit contre payant »
mais « étroit contre large ». Enfin, le local ajouté manuellement au tableau (prix marginal nul)
rappelle que le coût d'un modèle ne se lit pas seulement dans un catalogue : il se calcule
électricité, matériel et latence compris. Les prix affichés sont ceux **publiés à l'instant de
l'exécution** ; le notebook les relit à chaque passage plutôt que de les figer.

## 2. Cent questions à réponse vérifiable

Mesurer un routeur exige de **noter les réponses automatiquement** — un jury humain serait plus
lent que les modèles qu'il juge. On génère donc une banque de questions arithmétiques en
français, chaque question portant sa réponse exacte calculée par le générateur lui-même : la
note est une égalité numérique stricte, sans juge ni ambiguïté. C'est ce qui rendra les
comparaisons des sections suivantes **falsifiables** : deux stratégies sur la même banque ne
diffèrent que par leurs décisions, pas par l'étalon.

La graine est fixée : la même banque sert à tous les modèles du notebook, deux exécutions
produisent les mêmes questions — et la même graine servira aux notebooks suivants (caches,
exploitation), si bien qu'une justesse mesurée ici se compare à celle-là sans biais de tirage.
Les cinq types couvrent une échelle de difficulté grossière, de l'addition triviale au problème
à deux étapes qui exige de composer deux opérations : c'est **la variable expérimentale** que
les baselines vont mesurer et que le routeur essaiera de capter.

In [4]:
def question_addition(rnd):
    a, b = rnd.randint(120, 980), rnd.randint(120, 980)
    q = f"Un entrepôt reçoit {a} colis le matin, puis {b} colis l'après-midi. Combien de colis au total ?"
    return q, a + b

def question_soustraction(rnd):
    a, b = rnd.randint(400, 990), rnd.randint(100, 390)
    q = f"Un magasin possède {a} unités de stock et en vend {b} dans la journée. Combien reste-t-il d'unités ?"
    return q, a - b

def question_produit(rnd):
    a, b = rnd.randint(12, 96), rnd.randint(12, 96)
    q = f"Une imprimerie produit {a} affiches par série et fabrique {b} séries. Combien d'affiches au total ?"
    return q, a * b

def question_pourcentage(rnd):
    p = rnd.choice([10, 15, 20, 25, 30, 40, 50, 75])
    total = rnd.randrange(200, 1000, 4)
    q = f"Un service compte {total} dossiers et {p}% sont traités en priorité. Combien de dossiers prioritaires ?"
    return q, total * p // 100

def question_deux_etapes(rnd):
    a, b, c = rnd.randint(15, 80), rnd.randint(3, 12), rnd.randint(20, 90)
    q = (f"Une équipe produit {a} pièces par heure pendant {b} heures, puis {c} pièces de plus "
         f"en heures supplémentaires. Combien de pièces au total ?")
    return q, a * b + c

GENES = [question_addition, question_soustraction, question_produit,
         question_pourcentage, question_deux_etapes]
GENE_NOMS = [g.__name__.replace("question_", "") for g in GENES]

rnd = random.Random(20260928)   # graine fixée : banque identique à chaque exécution
BANQUE = []
for i in range(100):
    g = GENES[i % len(GENES)]   # répartition égale : 20 questions par type
    texte, reponse = g(rnd)
    BANQUE.append({"id": i, "type": GENE_NOMS[i % len(GENES)],
                   "question": texte, "reponse": reponse})

print(f"Banque : {len(BANQUE)} questions, {len(GENES)} types")
for q in BANQUE[:5] + BANQUE[-2:]:
    print(f"  [{q['id']:3d}] ({q['type']:14s}) {q['question']}  ->  {q['reponse']}")


Banque : 100 questions, 5 types
  [  0] (addition      ) Un entrepôt reçoit 823 colis le matin, puis 682 colis l'après-midi. Combien de colis au total ?  ->  1505
  [  1] (soustraction  ) Un magasin possède 686 unités de stock et en vend 116 dans la journée. Combien reste-t-il d'unités ?  ->  570
  [  2] (produit       ) Une imprimerie produit 39 affiches par série et fabrique 24 séries. Combien d'affiches au total ?  ->  936
  [  3] (pourcentage   ) Un service compte 560 dossiers et 75% sont traités en priorité. Combien de dossiers prioritaires ?  ->  420
  [  4] (deux_etapes   ) Une équipe produit 40 pièces par heure pendant 8 heures, puis 85 pièces de plus en heures supplémentaires. Combien de pièces au total ?  ->  405
  [ 98] (pourcentage   ) Un service compte 472 dossiers et 20% sont traités en priorité. Combien de dossiers prioritaires ?  ->  94
  [ 99] (deux_etapes   ) Une équipe produit 53 pièces par heure pendant 4 heures, puis 74 pièces de plus en heures supplémentaires. Com

### Lecture de la banque

Cent questions, cinq types, une graine fixée (**20260928** — la même que les notebooks 9d et 9e :
les justesses mesurées ici se comparent aux leurs sans biais de tirage). L'échelle de difficulté
est grossière mais réelle : l'addition à un chiffre (`823 + 682 -> 1505` en question 0) est
triviale ; le produit à deux facteurs (`39 x 24 -> 936` en question 2) demande plus ; le problème
à deux étapes (`40 pieces x 8 h + 85 -> 405`) exige de **composer** deux opérations avant de
répondre — c'est lui qui crééra l'écart décisif entre modèles à la section suivante. Deux
propriétés font la valeur de cette banque : chaque question **porte sa réponse exacte** calculée
par le générateur (la note est une égalité numérique stricte — aucun juge, aucune ambiguïté, pas
de biais de correction), et la graine rend la banque **identique à chaque exécution**. Mesurer un
routeur exige précisément cela : un étalon fixe sur lequel comparer des stratégies, pas des
questions qui changent entre deux mesures.
Cette banque n'est pas un quiz : c'est un **étalon d'étalonnage partagé**. La même graine
sert au notebook suivant (caches) et à celui d'exploitation (budget, canary) — une justesse
mesurée ici, une économie mesurée là-bas se lisent sur le même tirage de questions, et la
comparaison entre notebooks devient une comparaison entre mécanismes, toutes choses égales par
ailleurs.

## 3. Baselines : chaque modèle face à la banque

Avant de router, il faut connaître les extrêmes : que vaut chaque modèle seul, sur toute la
banque ? Un seul code d'appel parle aux deux fournisseurs — Ollama et OpenRouter exposent
tous deux l'API compatible OpenAI, et c'est ce **standard d'interface** qui rendra le repli
fournisseur possible à la section 6 (un maillon remplaçable est un maillon dont l'API est la
même). La réponse est extraite par expression régulière (dernier nombre du texte rendu) et la
note compare à la réponse exacte du générateur.

Les deux passes sont menées séparément, pour une raison qui deviendra un résultat : le tableau
**par type de question** compte plus que la note globale — c'est lui qui dira si un modèle
domine partout (et alors le routage est sans objet) ou si chacun a son territoire (et alors le
choix par question a de la valeur). La latence est mesurée en passant, car elle est la
troisième monnaie du verdict final.
Un choix d'instrument à noter avant les passes : la température est nulle et chaque modèle
garde ses conditions d'appel de production (pas d'avantage de banc fabriqué pour l'occasion).
Les deux modèles sont donc évalués **tels qu'ils serviraient**, et l'écart mesuré est l'écart
qu'un routeur pourrait réellement acheter.

In [5]:
from openai import OpenAI

def client_local():
    return OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")

def client_openrouter():
    assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY absente du .env"
    return OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

def extraire_nombre(texte):
    """Dernier entier du texte rendu (le modèle conclut en général par la valeur)."""
    nombres = re.findall(r"-?\d+", texte)
    return int(nombres[-1]) if nombres else None

SYSTEME = ("Tu es un assistant de calcul. Réponds en une seule phrase courte et termine par la "
           "valeur numérique exacte, sans unité.")

def poser(client, modele, question, max_retries=3, delai_min=0.0):
    """Un appel, avec retry simple sur erreur. Renvoie (réponse_extraite, texte, latence_s).

    ``delai_min`` espace les appels pour respecter le quota du fournisseur : le terrain gratuit
    du proxy est limité à 30 requêtes/minute sur gemma-4-31b (mesuré : 429 au-delà). La latence
    rendue est celle de l'appel, hors délai d'espacement."""
    for tentative in range(max_retries + 1):
        try:
            t0 = time.perf_counter()
            rep = client.chat.completions.create(
                model=modele,
                messages=[{"role": "system", "content": SYSTEME},
                          {"role": "user", "content": question}],
                temperature=0.0, max_tokens=200,
            )
            dt = time.perf_counter() - t0
            texte = rep.choices[0].message.content or ""
            if delai_min and dt < delai_min:
                time.sleep(delai_min - dt)   # espacement anti-quota, hors mesure de latence
            return extraire_nombre(texte), texte, dt
        except Exception as e:
            if tentative == max_retries:
                raise
            # 429 = quota fournisseur : attendre la fenêtre de quota avant de retenter
            if "429" in str(e) or "RateLimit" in type(e).__name__:
                time.sleep(20.0 * (tentative + 1))
            else:
                time.sleep(1.5 * (tentative + 1))


In [6]:
def evaluer_banque(client, modele, banque=BANQUE, delai_min=0.0):
    """Fait passer la banque à un modèle. Renvoie la liste des résultats par question."""
    resultats = []
    for q in banque:
        pred, texte, dt = poser(client, modele, q["question"], delai_min=delai_min)
        resultats.append({**q, "pred": pred, "correct": pred == q["reponse"],
                          "latence": dt, "texte": (texte or "")[:120]})
    return resultats

def bilan(resultats, nom):
    n = len(resultats)
    acc = 100.0 * sum(r["correct"] for r in resultats) / n
    lat = statistics.mean(r["latence"] for r in resultats)
    print(f"{nom:34s} exactitude {acc:5.1f}%  latence moy. {lat:5.2f}s  (n={n})")
    return {"nom": nom, "accuracy": acc, "latence": lat}


### Passage 1 : le modèle local, gratuit

Les 100 questions passent d'abord au modèle local. C'est la mesure de référence du coût zéro :
ce que le matériel présent offre sans déboursé un centime.


In [7]:
t_debut = time.perf_counter()
res_local = evaluer_banque(client_local(), LOCAL_ID)
duree_local = time.perf_counter() - t_debut
bil_local = bilan(res_local, LOCAL_KEY)
print(f"passe complète en {duree_local:.0f} s")
par_type = {}
for r in res_local:
    par_type.setdefault(r["type"], []).append(r["correct"])
for t, cors in sorted(par_type.items()):
    print(f"  {t:14s} {100.0 * sum(cors) / len(cors):5.1f}% correct")


local/qwen2.5:7b                   exactitude  65.0%  latence moy.  0.19s  (n=100)
passe complète en 20 s
  addition        95.0% correct
  deux_etapes      5.0% correct
  pourcentage     60.0% correct
  produit         65.0% correct
  soustraction   100.0% correct


### Passage 2 : un grand modèle du proxy, gratuit lui aussi

Le catalogue expose un grand modèle gratuit. Le passer sur la même banque donne la deuxième
référence : celle du coût zéro distant — gratuit en dollars, payé en latence réseau et en
dépendance.


In [8]:
GEMMA = "google/gemma-4-31b-it:free"
t_debut = time.perf_counter()
res_gemma = evaluer_banque(client_openrouter(), GEMMA, delai_min=2.5)
duree_gemma = time.perf_counter() - t_debut
bil_gemma = bilan(res_gemma, GEMMA)
print(f"passe complète en {duree_gemma:.0f} s")
par_type = {}
for r in res_gemma:
    par_type.setdefault(r["type"], []).append(r["correct"])
for t, cors in sorted(par_type.items()):
    print(f"  {t:14s} {100.0 * sum(cors) / len(cors):5.1f}% correct")


google/gemma-4-31b-it:free         exactitude  88.0%  latence moy.  1.34s  (n=100)
passe complète en 251 s
  addition       100.0% correct
  deux_etapes    100.0% correct
  pourcentage     40.0% correct
  produit        100.0% correct
  soustraction   100.0% correct


In [9]:
import pandas as pd

comparatif_types = pd.DataFrame({
    t: {
        LOCAL_KEY: 100.0 * sum(r["correct"] for r in res_local if r["type"] == t) / 20,
        GEMMA: 100.0 * sum(r["correct"] for r in res_gemma if r["type"] == t) / 20,
    } for t in GENE_NOMS
})
comparatif_types.index.name = "type de question"
comparatif_types.round(1)


,addition,soustraction,produit,pourcentage,deux_etapes
type de question,,,,,
local/qwen2.5:7b,95.0,100.0,65.0,60.0,5.0
google/gemma-4-31b-it:free,100.0,100.0,100.0,40.0,100.0


### Lecture des baselines

La table croisée est la matière première du routeur, et elle dit trois choses. **Un modèle
échoue là où l'autre réussit** : le local tombe à **5.0 %** sur les problèmes à deux étapes
contre **100.0 %** pour le distant — une faille totale sur le type le plus composé. **Le local
domine quelque part** : **60.0 %** contre **40.0 %** sur les pourcentages — le petit modèle
bat le grand sur un type entier, résultat contre-intuitif mais mesuré. Entre les deux,
l'addition est quasi-égalité (**95.0 %** contre **100.0 %**) et le produit penche nettement
distant (**65.0 %** contre **100.0 %**). Aucun modèle ne domine partout : **c'est exactement la
condition pour que le choix par question ait de la valeur** — un routeur qui enverrait chaque
type au modèle qui le réussit peut dépasser les deux baselines globales. La suite du notebook
teste précisément cette promesse, d'abord par une règle écrite d'avance (section 4), puis par
une décision après réponse (section 5). Les latences des passes complètes (20 s contre 251 s
pour cent questions) complètent le tableau : le dollar n'est pas la seule monnaie.

## 4. Routage par politique : coût × difficulté × budget

Le routeur le plus simple est une **règle écrite avant l'expérience** : estimer la difficulté
de la question, comparer au budget, choisir le modèle. Trois ingrédients donc — un estimateur
de difficulté, un budget par requête, un catalogue de prix — et la décision se prend **sans
avoir vu la réponse**, sur ce qu'on sait de la question avant de la poser. C'est toute la
différence avec la cascade de la section 5, qui décidera après avoir vu une première réponse.

L'estimateur retenu lit le type de question déclaré par le générateur : une information
**parfaite**, qu'un système réel n'aurait pas (il devrait inférer la difficulté du texte, et
ses erreurs feraient partie du coût). La mesure qui suit borne donc par le haut ce qu'un
routeur déployé obtiendrait — c'est un étalonnage du mécanisme, pas une promesse de production.
Le budget, lui, plafonne ce que le distant peut coûter par requête : sur ce terrain de modèles
gratuits il restera inactif, mais il est là pour montrer où la garde se mettrait.

In [10]:
def difficulte_estimee(q):
    """Score de difficulté 0-3, calculé SANS la réponse — ce que le routeur sait en production."""
    base = {"addition": 0, "soustraction": 1, "pourcentage": 1,
            "produit": 2, "deux_etapes": 3}
    return base.get(q["type"], 1)

class RouteurPolitique:
    """Route statique : facile -> local, difficile -> proxy. Budget plafonné en $/1k tokens."""
    def __init__(self, seuil_difficulte=2, budget_1k=0.01):
        self.seuil = seuil_difficulte
        self.budget = budget_1k

    def choisir(self, q):
        if difficulte_estimee(q) < self.seuil:
            return LOCAL_KEY, client_local(), LOCAL_ID
        # Vérifie que le modèle distant respecte le budget (prix in+out pour ~1k tokens)
        px = catalogue_prix[GEMMA]
        cout_1k = (px["in"] + px["out"]) * 1000 / 1e6
        if cout_1k > self.budget:
            return LOCAL_KEY, client_local(), LOCAL_ID
        return GEMMA, client_openrouter(), GEMMA

routeur = RouteurPolitique(seuil_difficulte=2, budget_1k=0.01)

# Où va chaque question ?
destinations = {}
for q in BANQUE:
    dest, _, _ = routeur.choisir(q)
    destinations.setdefault(dest, []).append(q["id"])
for dest, ids in destinations.items():
    print(f"{dest:38s} {len(ids):3d} questions")


local/qwen2.5:7b                        60 questions
google/gemma-4-31b-it:free              40 questions


In [11]:
def evaluer_routeur(routeur, banque=BANQUE):
    resultats = []
    for q in banque:
        dest, client, modele = routeur.choisir(q)
        pred, texte, dt = poser(client, modele, q["question"],
                                delai_min=2.5 if dest == GEMMA else 0.0)
        resultats.append({**q, "pred": pred, "correct": pred == q["reponse"],
                          "latence": dt, "modele": dest})
    return resultats

res_route = evaluer_routeur(routeur)
bil_route = bilan(res_route, "routeur politique")
utilisations = {}
for r in res_route:
    utilisations.setdefault(r["modele"], [0, 0])
    utilisations[r["modele"]][0] += 1
    utilisations[r["modele"]][1] += r["correct"]
for dest, (n, nc) in utilisations.items():
    print(f"  {dest:38s} {n:3d} questions, {100.0 * nc / n:5.1f}% correct")


routeur politique                  exactitude  91.0%  latence moy.  1.06s  (n=100)
  local/qwen2.5:7b                        60 questions,  85.0% correct
  google/gemma-4-31b-it:free              40 questions, 100.0% correct


### Lecture du routage

Le verdict se lit sur une seule ligne : **91.0 %** d'exactitude — **au-dessus des deux
baselines** mesurées en section 3 — pour une latence moyenne de **1.06 s**. Le routeur a envoyé
**60 questions** au local et **40** au grand modèle, et chaque moitié a été servie par le modèle
qui la réussit le mieux : le distant répond **100.0 %** correct sur sa part (les types durs,
produit et deux-étapes), le local **85.0 %** sur la sienne (les types faciles — son 85 % local
reflète ses ratés sur les pourcentages vus en section 3). L'arithmétique est cohérente : 51
réussites locales + 40 distantes = les 91 du total. Deux limites à lire en même temps que le
chiffre : l'estimateur de difficulté lit le **type déclaré** par le générateur — une information
parfaite qu'un système réel n'a pas, la mesure borne donc par le haut ce qu'un routeur déployé
obtiendrait ; et le budget par requête ($0.01 pour ~1k tokens) n'a rien filtré ici, puisque le
grand modèle retenu est gratuit. C'est un plafond de sécurité, pas une contrainte active — sur
un catalogue payant, il deviendrait le garde-fou décisif.

## 5. Cascade petit-vers-grand sur seuil de confiance

Le routage statique décide **avant** de voir la réponse ; la cascade décide **après** : le
modèle économique répond d'abord, déclare lui-même sa confiance, et une confiance sous le seuil
déclenche une seconde demande au grand modèle. Le mécanisme a une élégance économique — on ne
paie le grand modèle que pour les questions qui le justifient — et une condition de validité :
que la **confiance déclarée veuille dire quelque chose**.

C'est ce que cette section mesure explicitement, dans cet ordre : d'abord la **calibration**
(combien de réponses fausses le petit modèle déclare-t-il avec assurance ? combien de justes
escaladerait-il inutilement ?), ensuite seulement la cascade au seuil choisi. Inverser l'ordre
— choisir un seuil parce qu'il « semble raisonnable » puis mesurer — est l'erreur de
conception que la table de calibration ci-dessous rend visible. Les deux taux qui valident ou
cassent le seuil : le **faux succès** (faux et sûr de lui : jamais récupéré) et l'escalade
inutile (juste mais doutant : payé deux fois).

In [12]:
SYSTEME_CONFIANCE = ("Tu es un assistant de calcul. Réponds sur une seule ligne, au format exact : "
                     "REPONSE: <valeur> CONFIANCE: <nombre entre 0 et 1>")

def poser_avec_confiance(client, modele, question):
    t0 = time.perf_counter()
    rep = client.chat.completions.create(
        model=modele,
        messages=[{"role": "system", "content": SYSTEME_CONFIANCE},
                  {"role": "user", "content": question}],
        temperature=0.0, max_tokens=200,
    )
    dt = time.perf_counter() - t0
    texte = (rep.choices[0].message.content or "").replace(" ", "")
    m_val = re.search(r"REPONSE:\"?(-?\d+)", texte)
    m_conf = re.search(r"CONFIANCE:\"?([01]?[.,]?\d*)", texte)
    val = int(m_val.group(1)) if m_val else None
    try:
        conf = float(m_conf.group(1).replace(",", ".")) if m_conf and m_conf.group(1) else 0.5
    except ValueError:
        conf = 0.5
    return val, min(max(conf, 0.0), 1.0), dt

# Phase de calibration : on mesure la confiance déclarée sur la banque, SANS escalader
calibration = []
for q in BANQUE:
    val, conf, dt = poser_avec_confiance(client_local(), LOCAL_ID, q["question"])
    calibration.append({**q, "pred": val, "correct": val == q["reponse"], "confiance": conf})

conf_justes = [c["confiance"] for c in calibration if c["correct"]]
conf_fausses = [c["confiance"] for c in calibration if not c["correct"]]
n_fausses = len(conf_fausses)
faux_succes = sum(1 for c in calibration if not c["correct"] and c["confiance"] >= 0.8)
print(f"Confiance moyenne — justes : {statistics.mean(conf_justes):.2f}  "
      f"fausses : {statistics.mean(conf_fausses) if conf_fausses else float('nan'):.2f}")
print(f"Faux succès (réponse fausse, confiance >= 0.8) : {faux_succes} / {n_fausses} réponses fausses")


Confiance moyenne — justes : 1.00  fausses : 1.00
Faux succès (réponse fausse, confiance >= 0.8) : 21 / 21 réponses fausses


In [13]:
df_cal = pd.DataFrame(calibration)
table_calibration = pd.crosstab(
    pd.cut(df_cal["confiance"], [0, .4, .6, .8, 1.0], include_lowest=True),
    df_cal["correct"],
    rownames=["confiance déclarée"], colnames=["réponse exacte"])
table_calibration


réponse exacte,False,True
confiance déclarée,,
"(0.8, 1.0]",21,79


### Lecture de la calibration

La table croisée dit tout, et elle dit une catastrophe : **toutes** les réponses vivent dans la
ligne `(0.8, 1.0]` — **21** fausses et **79** exactes, aucune confidence basse déclarée.
Autrement dit : le modèle local ne **dit jamais** qu'il doute. La confiance moyenne mesurée à la
cellule précédente est maximale pour les justes **comme pour les fausses** — le score n'a
aucun pouvoir discriminant, et les **21 réponses fausses sont toutes déclarées à 0.8 ou plus**
(des faux succès intégraux, mesurés 21/21). La conséquence est mécanique : **aucun seuil ne peut
récupérer ces 21 erreurs**, car aucune ne se signalera jamais pour escalader. C'est le plafond
d'exactitude de toute cascade bâtie sur cette confiance déclarée — et la démonstration la plus
utile de cette section : **la calibration se mesure avant de choisir un seuil**, jamais après.
Un modèle sûr de lui à tort convertit un mécanisme d'économie (escalader rarement) en
incapacité structurelle (escalader jamais).
La seconde classe d'erreur, symétrique, est ici invisible par construction — et c'est une
information : une escalade inutile (réponse juste doubtee) se lirait dans les lignes basses de
la table, qui sont **vides**. Le modèle ne doute ni à tort ni à raison : il ne doute jamais.
Une cascade bien portante vit dans l'équilibre entre ces deux taux ; celle-ci vit dans un
desert de doute, où le seuil n'a aucune matière sur laquelle mordre.

In [14]:
SEUIL_CASCADE = 0.6   # choisi depuis la table de calibration ci-dessus

def cascade(question):
    """Local d'abord ; escalade vers le grand modèle si confiance < seuil."""
    val, conf, dt1 = poser_avec_confiance(client_local(), LOCAL_ID, question)
    if conf >= SEUIL_CASCADE:
        return val, f"local(conf={conf:.2f})", dt1, False
    val2, texte2, dt2 = poser(client_openrouter(), GEMMA, question, delai_min=2.5)
    return val2, "cascade->gemma", dt1 + dt2, True

res_cascade = []
n_escalades = 0
for q in BANQUE:
    val, origine, dt, escalade = cascade(q["question"])
    n_escalades += escalade
    res_cascade.append({**q, "pred": val, "correct": val == q["reponse"],
                        "latence": dt, "escalade": escalade})

bil_cascade = bilan(res_cascade, f"cascade (seuil {SEUIL_CASCADE})")
print(f"escalades vers le grand modèle : {n_escalades}/{len(BANQUE)}")


cascade (seuil 0.6)                exactitude  79.0%  latence moy.  0.32s  (n=100)
escalades vers le grand modèle : 0/100


### Lecture de la cascade

Le chiffre le plus instructif est le dernier : **escalades 0/100**. La cascade ne s'est jamais
déclenchée — la confiance déclarée étant toujours au-dessus du seuil 0.6, chaque question est
restée au local. En théorie, elle a donc dégénéré en « local seul » ; en pratique, elle rend
**79.0 %** pour **0.32 s** de latence moyenne — mieux que la passe libre du local en section 3
(65.0 %), sans avoir payé un seul appel distant. D'où vient ce gain de 14 points alors que le
même modèle sert les mêmes questions ? Du **format contraint** : la cellule de calibration
exige une réponse à une ligne (`REPONSE: <valeur> CONFIANCE: <nombre>`), ce qui discipline
l'extraction et le comportement du modèle par rapport à la réponse libre de la section 3. La
leçon honnête est double : le gain mesuré ici vient du prompt, pas de la cascade ; et **le seuil
n'a servi à rien parce que la calibration ne le portait pas** — exactement ce que la section
précédente avait prédit. La cascade n'est pas réfutée pour autant : sur un modèle mieux calibré,
le même mécanisme escaladerait les vrais doutes. Elle est simplement inopérante tant que la
confiance déclarée ne veut rien dire.

## 6. Repli fournisseur sur panne simulée

Dernière brique de production : le proxy tombe. On simule la panne proprement — une URL valide
en apparence pointant vers un **port fermé** (le service arrêté d'un faux proxy), si bien que
le code d'appel, lui, reste exactement celui de production : la panne frappe l'infrastructure,
pas le notebook. La chaîne de repli essaie le maillon mort, puis bascule sur le proxy réel,
puis sur l'inférence locale en dernier recours.

On mesure les deux faces du repli, car il n'est gratuit ni dans un sens ni dans l'autre : ce
qu'il **sauve** (requêtes servies malgré la panne — la disponibilité) et ce qu'il **coûte** (la
latence des tentatives mortes payées avant chaque bascule). L'échantillon est réduit à trente
questions : le motif — un échec, une bascule, un service — se répète à l'identique, et ce qui
compte est sa mesure, pas son volume. La section suivante réunit toutes les stratégies sur la
même table.
L'ordre de la chaîne est lui-même une décision : le proxy réel passe avant le local parce
qu'il est disponible **et** plus juste — le repli dégrade vers le local, qui reste le dernier
recours (étroit, mais autonome du réseau). Inverser l'ordre reviendrait à servir local par
défaut et à ne toucher le proxy que si Ollama tombait aussi : une stratégie différente, pas
nécessairement meilleure — à arbitrer en fonction de ce que chaque maillon coûte et réussit,
exactement comme à la section 4.

In [15]:
PROXY_MORT = "http://127.0.0.1:11435/v1"   # port fermé : la panne simulée

def poser_avec_repli(question, chaine):
    """Essaie chaque (client, modèle) de la chaîne jusqu'à une réponse. Renvoie (pred, essais)."""
    essais = []
    for client, modele in chaine:
        try:
            t0 = time.perf_counter()
            pred, texte, _ = poser(client, modele, question, max_retries=0,
                                     delai_min=2.5 if modele == GEMMA else 0.0)
            essais.append((modele, "ok", time.perf_counter() - t0))
            return pred, essais
        except Exception as e:
            essais.append((modele, f"échec: {type(e).__name__}", time.perf_counter() - t0))
            continue
    return None, essais

CHAINE = [
    (OpenAI(base_url=PROXY_MORT, api_key="simulation"), GEMMA),  # simule le proxy en panne
    (client_openrouter(), GEMMA),                                 # proxy réel
    (client_local(), LOCAL_ID),                                   # dernier recours local
]

res_repli = []
for q in BANQUE[:30]:   # échantillon : la démonstration suffit, le motif est répétitif
    pred, essais = poser_avec_repli(q["question"], CHAINE)
    res_repli.append({**q, "pred": pred, "correct": pred == q["reponse"],
                      "essais": essais, "n_essais": len(essais)})

servies = sum(1 for r in res_repli if r["pred"] is not None)
justes = sum(1 for r in res_repli if r["correct"])
essais_morts = [r["essais"][0][2] for r in res_repli if r["n_essais"] > 1]
print(f"requêtes servies malgré la panne du premier maillon : {servies}/{len(res_repli)}")
print(f"dont exactes : {justes}/{len(res_repli)}")
if essais_morts:
    print(f"surcoût de latence de la tentative morte : {statistics.mean(essais_morts):.2f} s en moyenne")


requêtes servies malgré la panne du premier maillon : 30/30
dont exactes : 27/30
surcoût de latence de la tentative morte : 7.45 s en moyenne


### Lecture du repli

**30 requêtes sur 30 servies** malgré le premier maillon mort par construction : la
disponibilité est totale sur l'échantillon, et c'est précisément ce que la chaîne de repli
achète. Les **27/30 exactes** correspondent au niveau du maillon qui a réellement répondu (le
proxy réel, deuxième de chaîne) — le repli ne dégrade pas la qualité, il déplace la charge. Le
prix est mesuré sans fard : **7.45 s de tentative morte en moyenne par requête**, le temps
d'attendre l'échec du port fermé avant de basculer. Sur trente questions, c'est une gêne ; sur
un service à forte charge, c'est un **triplement de latence perçue** pendant toute la panne —
et la raison pour laquelle une vraie chaîne de production intercale une **sonde de santé**
(découvrir la panne sans la payer à chaque requête) voire un **circuit breaker** (arrêter
d'essayer un maillon déclaré ouvert, pendant une fenêtre de refroidissement — c'est l'objet de
l'exercice 3). Le motif démontré : le repli convertit une panne totale en dégradation de
latence, à condition d'accepter de la mesurer.

## 7. Synthèse : ce que chaque stratégie achète

Les passes de ce notebook sont réunies ici, sur la même table et le même étalon : chaque ligne
est une stratégie **réellement exécutée** sur la même banque de cent questions, aux colonnes
les trois monnaies du routage — exactitude, latence, coût. Les baselines de la section 3, le
routeur politique de la section 4 et la cascade de la section 5 s'y lisent d'un seul regard.

La table n'est pas un classement : aucune stratégie ne domine les trois colonnes, et c'est
précisément la leçon de conclusion. Le coût se calcule depuis les prix du catalogue relevés en
section 1 et les volumes d'appels mesurés — escalades du routeur et de la cascade comprises,
car une stratégie ne paie que ce qu'elle envoie réellement au payant.
Comment lire cette table : ligne à ligne, une stratégie ; colonne par colonne, une monnaie.
Le bon réflexe est de choisir d'abord **la colonne qui compte pour votre service** (un
assistant temps réel arbitre sur la latence, un correcteur d'exercices sur l'exactitude), puis
de regarder ce que cette stratégie paie sur les deux autres. C'est la lecture que fait la
conclusion — et la raison pour laquelle aucune ligne n'est « la meilleure » dans l'absolu.

In [16]:
def cout_usd(n_questions, modele_key):
    px = catalogue_prix[modele_key]
    # estimation : ~60 tokens d'entrée, ~30 de sortie par question (messages courts du notebook)
    return n_questions * (60 * px["in"] + 30 * px["out"]) / 1e6

synthese = pd.DataFrame([
    {**bil_local,   "coût USD": cout_usd(len(res_local), LOCAL_KEY),   "stratégie": "local seul"},
    {**bil_gemma,   "coût USD": cout_usd(len(res_gemma), GEMMA),      "stratégie": "grand distant seul"},
    {**bil_route,   "coût USD": cout_usd(n_escalades_route := sum(1 for r in res_route if r["modele"] == GEMMA), GEMMA), "stratégie": "routage politique"},
    {**bil_cascade, "coût USD": cout_usd(n_escalades, GEMMA),         "stratégie": "cascade"},
])[["stratégie", "accuracy", "latence", "coût USD"]]
synthese.round(4)


,stratégie,accuracy,latence,coût USD
0,local seul,65.0,0.1948,0.0
1,grand distant seul,88.0,1.3418,0.0
2,routage politique,91.0,1.0572,0.0
3,cascade,79.0,0.3202,0.0


### Lecture de la synthèse

Quatre lignes, trois monnaies, et aucune ne domine partout. Le **routage politique** achète le
sommet d'exactitude (**91.0**) au prix de la latence du distant (**1.0572 s** en moyenne — il
envoie 40 % du trafic vers le grand modèle). La **cascade** occupe une position intermédiaire
originale : **79.0** d'exactitude à **0.3202 s** — presque la latence du local, 12 points
au-dessus de sa passe libre. Le **local seul** reste la référence de coût et de latence
(**0.1948 s**) au prix du plancher d'exactitude (**65.0**), et le **grand distant seul**
(**88.0** à **1.3418 s**) est battu par le routage qu'il alimente pourtant. La colonne **coût
USD vaut 0.0 partout** — conséquence directe du choix d'un grand modèle gratuit au catalogue :
sur CE terrain, le dollar ne discrimine aucune stratégie. Ce n'est pas un défaut de la mesure,
c'est une de ses conclusions : sur un catalogue payant, les mêmes quatre lignes montreraient le
routage achetant aussi de l'argent. La leçon générale tient en une phrase : **le routage est un
arbitrage entre monnaies, pas un classement** — choisir, c'est dire quelle monnaie on achète et
laquelle on accepte de payer.

## Exercices

Les trois exercices reprennent les briques du notebook là où la version de cours s'arrête
volontairement — chacun durcit un mécanisme mesuré plus haut.

### Exercice 1 : budget par requête durci

Le `RouteurPolitique` vérifie un budget pour ~1k tokens. Durcissez-le : le routeur doit refuser
tout modèle dont le coût estimé dépasse un budget exprimé en **dollars par requête**, estimé
depuis les `max_tokens` et la longueur constatée des questions. Documentez ce que le budget
durci change aux destinations.

### Exercice 2 : cascade à trois niveaux

Ajoutez un niveau intermédiaire (par ex. `mistralai/mistral-nemo`, payant quasi gratuit) entre le
local et le grand : local → intermédiaire → grand. Quelle exactitude pour quel taux d'escalade,
par rapport à la cascade à deux niveaux ?

### Exercice 3 : circuit breaker

Le repli du section 6 essaie le maillon mort à chaque requête. Implémentez un circuit breaker :
après N échecs consécutifs, le maillon est déclaré ouvert pendant une durée de refroidissement,
et les requêtes vont directement au suivant. Mesurez le gain de latence sur les 30 mêmes
questions.

In [17]:
# Exercice 1 -- budget par requête durci
# TODO étudiant : spécialiser RouteurPolitique en RouteurBudgetDure
# Indice : estimer tokens_entree ~ 60 + len(question)//4, tokens_sortie ~ max_tokens
# Étape 1 : ajouter un paramètre budget_par_requete (USD)
# Étape 2 : calculer le coût pire-cas par modèle du catalogue
# Étape 3 : aiguiller vers le premier modèle sous budget, du moins cher au plus cher
print("Exercice à compléter")
resultat_ex1 = None  # TODO étudiant : DataFrame des destinations sous budget durci


Exercice à compléter


In [18]:
# Exercice 2 -- cascade à trois niveaux
# TODO étudiant : généraliser cascade() en cascade_niveaux(question, niveaux, seuils)
# Indice : chaque niveau porte son propre seuil de confiance
# Étape 1 : définir la liste [(local, 0.6), (nemo, 0.75), (gemma, None)]
# Étape 2 : escalader tant que la confiance déclarée est sous le seuil du niveau
# Étape 3 : mesurer exactitude, escalades et latence, comparer à la cascade à deux niveaux
print("Exercice à compléter")
resultat_ex2 = None  # TODO étudiant : bilan de la cascade à trois niveaux


Exercice à compléter


In [19]:
# Exercice 3 -- circuit breaker
# TODO étudiant : implémenter la classe CircuitBreaker
# Indice : états fermé / ouvert / semi-ouvert, compteur d'échecs, horodatage d'ouverture
# Étape 1 : CircuitBreaker(seuil=3, refroidissement_s=30) avec méthode autoriser()
# Étape 2 : poser_avec_repli() consulte le breaker avant chaque maillon, enregistre succès/échec
# Étape 3 : rejouer les 30 questions et comparer la latence moyenne au section 6
print("Exercice à compléter")
resultat_ex3 = None  # TODO étudiant : latence moyenne avec et sans breaker


Exercice à compléter


## Conclusion

### Récapitulatif

| Brique | Ce qu'elle décide | Ce qu'elle coûte si mal réglée |
|---|---|---|
| Catalogue et prix | la matière première | décision sur des prix périmés |
| Banque vérifiable | la mesure | conclusions non falsifiables |
| Routage politique | **avant** la réponse | erreurs d'estimation de difficulté |
| Cascade | **après** la réponse | faux succès mal calibrés |
| Repli fournisseur | **pendant** la panne | tentative morte payée en latence |

### La lecture d'ensemble

Les trois briques formaient une progression : la politique a battu les deux baselines parce
qu'elle avait une information parfaite (le type déclaré) ; la cascade n'a rien escaladé parce
que la confiance déclarée n'était pas calibrée ; le repli a tout servi en payant chaque panne
en secondes. Aucun de ces verdicts n'est universel — chacun dépend d'une **condition mesurée**
(information de l'estimateur, calibration du modèle, fréquence des pannes), et c'est la
méthode qui se transfère : mesurer la condition avant de déployer le mécanisme. Le routage
n'est pas un classement de modèles mais un arbitrage de monnaies — exactitude, latence, coût,
disponibilité — et ce notebook a montré comment chaque arbitrage se chiffre sur un étalon fixe.

### La suite

Le notebook suivant (09d) garde la même banque et s'attaque aux **caches** : cache sémantique et
son taux de faux succès mesuré, cache de préfixe et son point d'équilibre. Le notebook 09e
referme l'arc côté exploitation : budget, alerte, shadow, canary et SLO sur le même terrain.

### Sources

- Catalogue et prix : `GET /v1/models` du proxy multi-fournisseurs (lectures de ce notebook)
- Terrain local : service d'inférence local, `GET /v1/models`

## Navigation

**Série GenAI Texte** | Suivant : [9d — Production Caches](09d_Production_Caches.ipynb) | [↑ Sommaire série](README.md)
